# سامانهٔ تشخیص پوششِ صورت — نسخهٔ ۴

نسخهٔ پایه + سه گامِ درخواستی. بقیهٔ الگوریتم **دست‌نخورده** است؛
هر تغییر در کد با `★ گام N` علامت خورده تا پیدا کردنش آسان باشد.

## سه تغییرِ این نسخه

| گام | چه شد | هزینه |
|---|---|---|
| **۵** | قفلِ سبز دیگر دائمی نیست — هر ۲ ثانیه یک بازبینی | ~۱.۷٪ حالتِ عادی |
| **۶** | حداقلِ اندازهٔ صورت + نمایشِ پیشرفتِ بررسی | صفر (کمتر هم می‌شود) |
| **۷** | رنگِ اسکلت = رنگِ وضعیتِ فرد | صفر |

## جدول تصمیم

| رنگ | برچسب | معنی |
|---|---|---|
| ⚪ خاکستری | `Analyzing... (2/3)` | دیده شده، در حالِ بررسی — عدد یعنی چند رأی جمع شده |
| ⚪ خاکستری | `Analyzing... (too far)` | دیده شده ولی هنوز خیلی دور است |
| 🟢 سبز | `Clear` | صورت باز — **هر ۲ ثانیه بازبینی می‌شود** |
| 🟠 نارنجی | `Medical Mask` | ماسک دارد ولی بالای صورت پیداست |
| 🔴 قرمز | `SUSPICIOUS - ALERT` | ماسک دارد و بالای صورت هم پوشیده است |

کادر، اسکلت و برچسب هر سه با همین رنگ کشیده می‌شوند.

> **قبل از شروع:** Runtime ← Change runtime type ← GPU

---
## ۰) نصب و راه‌اندازی

In [ ]:
!pip install -q ultralytics opencv-python-headless transformers torch torchvision pillow

### وارد کردن کتابخانه‌ها و انتخاب دستگاه

In [ ]:
import cv2, time, torch, os, json, numpy as np
from collections import deque
from PIL import Image
from ultralytics import YOLO
from transformers import AutoImageProcessor, SiglipForImageClassification

device = "cuda" if torch.cuda.is_available() else "cpu"
USE_HALF = device == "cuda"
print(f"🔧 Device: {device} | FP16: {USE_HALF}")

---
## ۱) مدل ژست — تشخیص فرد، اسکلت و ردیابی

یک مدل، سه کار: جعبهٔ هر فرد، ۱۷ کی‌پوینتِ COCO، و شناسهٔ پایدار برای
دنبال‌کردن هر نفر بین فریم‌ها (ByteTrack).

In [ ]:
# ---------------- 1) Single Unified Model: Person+Pose+Track ----------

# ★ B1 — مدلِ ژست حالا متغیر است، نه ثابت.
#
#   yolo11n-pose  سبک‌ترین. همان چیزی که تا حالا استفاده می‌شد.
#   yolo11s-pose  ★ پیش‌فرضِ جدید — کی‌پوینتِ دقیق‌تر، مخصوصاً برای
#                 سوژه‌های دور و مچ/دست. حدودِ ۲ برابر کندتر از n
#                 ولی چون کی‌پوینتِ بهتر یعنی کادرِ بهترِ صورت،
#                 روی کلِ زنجیره اثر مثبت دارد.
#   yolo26s-pose  نسلِ بعدی (منتشرشده ژانویهٔ ۲۰۲۶). NMS-free و با
#                 RLE برای مکان‌یابیِ دقیق‌ترِ کی‌پوینت. مقالهٔ رسمی
#                 تا +۷.۲ AP نسبت به YOLO11 روی COCO-pose گزارش کرده.
#
#   ⚠️ اگر به yolo26 سوییچ کردی: نحوهٔ کالیبراسیونِ *اطمینانِ*
#      کی‌پوینت‌ها ممکن است فرق کند، و گیتِ ما (FACE_CONF_TH) دقیقاً
#      روی همان اطمینان‌ها کار می‌کند. پس بعد از سوییچ حتماً یک بار
#      خروجی را نگاه کن؛ شاید لازم باشد ۰.۵ را کمی بالا/پایین ببری.
POSE_WEIGHTS = "yolo11s-pose.pt"
pose_model = YOLO(POSE_WEIGHTS)


NOSE, LEYE, REYE, LEAR, REAR = 0, 1, 2, 3, 4
LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST = 5, 6, 7, 8, 9, 10

# اسکلت فقط بالاتنه (سر، شونه، بازو، ساعد) - برای جلوه بصری
UPPER_BODY_SKELETON = [
    (LEYE, REYE), (NOSE, LEYE), (NOSE, REYE),
    (LEAR, LEYE), (REAR, REYE),
    (LSHOULDER, RSHOULDER),
    (LSHOULDER, LELBOW), (LELBOW, LWRIST),
    (RSHOULDER, RELBOW), (RELBOW, RWRIST),
    (NOSE, LSHOULDER), (NOSE, RSHOULDER),
]

---
## ۲) طبقه‌بندِ ماسک

مدلِ دوکلاسهٔ فاین‌تیون‌شده: `mask` یا `no_mask`. دسته‌ای کار می‌کند —
همهٔ صورت‌های یک فریم با هم به مدل می‌روند.

> یک بار این را اجرا کن و با `MASK_ID2LABEL` مقایسه کن:
> `print(mask_model.config.id2label)`

In [ ]:
# ---------------- 2) Mask Classifier (Pretrained, FP16) ----------------
MASK_MODEL_NAME = "prithivMLmods/Face-Mask-Detection"
mask_processor = AutoImageProcessor.from_pretrained(MASK_MODEL_NAME)
mask_model = SiglipForImageClassification.from_pretrained(MASK_MODEL_NAME).to(device).eval()
if USE_HALF:
    mask_model = mask_model.half()
MASK_ID2LABEL = {0: "mask", 1: "no_mask"}

@torch.no_grad()
def classify_mask_batch(face_list):
    if len(face_list) == 0:
        return []
    pil_imgs = [Image.fromarray(cv2.cvtColor(f, cv2.COLOR_BGR2RGB)) for f in face_list]
    inputs = mask_processor(images=pil_imgs, return_tensors="pt").to(device)
    if USE_HALF:
        inputs = {k: (v.half() if v.dtype == torch.float32 else v) for k, v in inputs.items()}
    logits = mask_model(**inputs).logits.float()
    probs = torch.nn.functional.softmax(logits, dim=1).cpu().numpy()
    out = []
    for p in probs:
        pred = int(np.argmax(p))
        out.append((MASK_ID2LABEL[pred], float(p[pred])))
    return out

---
## ۳) سنجهٔ پوست — تفکیکِ ماسکِ پزشکی از مشکوک

اگر کسی ماسکِ پزشکی زده، بالای صورتش پوست دیده می‌شود. اگر پوششِ
کامل داشته باشد، آنجا هم پوشیده است. `is_suspicious` نسبتِ پوست را
روی ۴۵٪ بالای برش می‌سنجد؛ زیر ۰.۱۲ یعنی مشکوک.

In [ ]:
# ---------------- 3) Skin-ratio heuristic (Medical vs Suspicious) -----
def skin_ratio(region_bgr):
    if region_bgr is None or region_bgr.size == 0:
        return 0.0
    hsv = cv2.cvtColor(region_bgr, cv2.COLOR_BGR2HSV)
    lower = np.array([0, 20, 40], dtype=np.uint8)
    upper = np.array([25, 180, 255], dtype=np.uint8)
    m = cv2.inRange(hsv, lower, upper)
    return float(np.count_nonzero(m)) / m.size

def is_suspicious(face_bgr):
    h, w, _ = face_bgr.shape
    upper = face_bgr[0:int(h * 0.45), :]
    return skin_ratio(upper) < 0.12

---
## ۴) تراز و برشِ صورت  ★ گام ۶

**گیتِ ورودی:** میانگینِ اطمینانِ بینی و دو چشم. زیرِ آستانه → `None`
→ آن فرد در آن فریم رأی نمی‌دهد. همین باعث می‌شود نیم‌رخ‌ها و
پشت‌به‌دوربین‌ها قضاوت نشوند.

**★ گام ۶ — `min_eye_dist`:** آستانهٔ فاصلهٔ دو چشم از ۳ به **۸**
رسید. قبلاً صورتی به عرضِ ~۱۰ پیکسل هم رأی می‌داد و آن رأی نویزِ
خالص بود. حالا فردِ دور همچنان **دیده و ردیابی می‌شود** — فقط تا
نزدیک‌تر نشده حکمی درباره‌اش صادر نمی‌شود.

In [ ]:
# ---------------- 4) Keypoint-based Face Visibility + Align + Crop ----
def align_and_crop_face(person_img, kxy, kconf, conf_th=0.5, min_eye_dist=8):
    nose_c, leye_c, reye_c = kconf[NOSE], kconf[LEYE], kconf[REYE]
    face_score = float((nose_c + leye_c + reye_c) / 3.0)
    if face_score < conf_th:
        return None, face_score

    leye, reye = kxy[LEYE], kxy[REYE]
    eye_dist = float(np.linalg.norm(np.array(leye) - np.array(reye)))
    # ★ گام ۶ — حداقلِ اندازهٔ صورت.
    #   قبلاً این عدد ۳ بود؛ یعنی صورتی به عرضِ ~۱۰ پیکسل هم رأی می‌داد و
    #   آن رأی عملاً نویزِ خالص بود. با ۸، فردِ دور دیده و ردیابی می‌شود
    #   ولی تا وقتی به‌قدرِ کافی نزدیک نشده، حکمی دربارهٔ او صادر نمی‌شود
    #   و در حالتِ «Analyzing» می‌ماند.
    if eye_dist < min_eye_dist:
        return None, face_score

    h, w = person_img.shape[:2]
    dy, dx = reye[1] - leye[1], reye[0] - leye[0]
    angle = np.degrees(np.arctan2(dy, dx))
    eye_center = ((leye[0] + reye[0]) / 2.0, (leye[1] + reye[1]) / 2.0)

    M = cv2.getRotationMatrix2D(eye_center, angle, 1.0)
    rotated = cv2.warpAffine(person_img, M, (w, h))

    half_w = eye_dist * 1.6
    top = eye_center[1] - eye_dist * 1.3
    bottom = eye_center[1] + eye_dist * 2.6

    x1, x2 = int(max(0, eye_center[0] - half_w)), int(min(w, eye_center[0] + half_w))
    y1, y2 = int(max(0, top)), int(min(h, bottom))
    if (x2 - x1) < 10 or (y2 - y1) < 10:
        return None, face_score

    return rotated[y1:y2, x1:x2], face_score

---
## ۵) ماشینِ حالت  ★ گام ۵

رأی‌ها روی چند فریم جمع می‌شوند تا رنگ‌ها چشمک نزنند.

| حالت | نرخِ بررسی |
|---|---|
| `fast` | هر فریم — تا ۳ رأی جمع شود |
| `focus` | هر ۲ فریم — فردِ ماسک‌دار زیرِ نظر |
| `locked` سبز | **هر ۶۰ فریم — ★ گام ۵** |

**★ گام ۵ — شکستنِ قفلِ سبز.** سناریو: کسی با صورتِ باز وارد می‌شود،
سبز قفل می‌شود، بعد داخل ماسک می‌کشد. با قفلِ دائمی سیستم دیگر هرگز
نگاهش نمی‌کرد.

حالا هر ۶۰ فریم (≈۲ ثانیه) یک بار بررسی می‌شود:

- رأی سبز آمد → ساعت صفر می‌شود، سبز می‌ماند
- رأی نارنجی/قرمز آمد → **قفل می‌شکند** و می‌رود به حالتِ فوکوس

**هزینه‌اش دقیقاً چقدر است؟** برای هر فردِ سبز، یک اجرای طبقه‌بند در
هر ۶۰ فریم به‌جای صفر. در حالتِ عادی هر فرد هر فریم بررسی می‌شد،
پس این یعنی **حدودِ ۱.۷٪** آن هزینه. عملاً رایگان.

عدد را می‌توانی عوض کنی: `state_mgr.GREEN_RECHECK_FRAMES = 90`

In [ ]:
# ---------------- 5) Smart State Manager (same logic as v4) -----------
COLORS = {"gray": (160, 160, 160), "green": (0, 200, 0),
          "orange": (0, 140, 255), "red": (0, 0, 255)}
LABELS = {"gray": "Analyzing...", "green": "Clear",
          "orange": "Medical Mask", "red": "SUSPICIOUS - ALERT"}

class TrackStateManager:
    def __init__(self):
        self.data = {}
        self.FAST_VOTES_NEEDED = 3
        self.FOCUS_INTERVAL = 2
        self.FOCUS_WINDOW = 4
        self.FOCUS_GREEN_NEEDED = 3
        # ★ گام ۵ — قفلِ سبز دیگر دائمی نیست.
        #   هر ۶۰ فریم (≈۲ ثانیه در ۳۰fps) یک بار دوباره بررسی می‌شود.
        #   هزینه: برای هر فردِ سبز، یک اجرای طبقه‌بند در هر ۶۰ فریم
        #   به‌جای صفر — یعنی حدودِ ۱.۷٪ حالتِ عادی. عملاً رایگان.
        self.GREEN_RECHECK_FRAMES = 60
        # ★ C2 — هر چند فریم یک بار، به تفکیکِ وضعیت
        self.CADENCE = {"red": 1, "orange": 3, "gray": 1}

    def ensure(self, tid):
        if tid not in self.data:
            self.data[tid] = {
                "mode": "fast", "locked": False, "votes": [],
                "focus_window": deque(maxlen=self.FOCUS_WINDOW),
                "color": "gray", "label": LABELS["gray"],
                "is_new": True,        # برای افکت نمایشی: آیا تازه معرفی شده؟
                "just_finalized": None,# برای افکت نمایشی: آیا همین الان قفل نهایی گرفته؟
                "conf": 0.0,           # ★ A3: اطمینانِ حکمِ فعلی (۰..۱)
                "locked_frame": 0,     # ★ گام ۵: آخرین فریمی که قفل/بازبینی شد
                "checks": 0,           # ★ گام ۶: چند بار تا حالا بررسی شده
            }
        return self.data[tid]

    def should_analyze(self, tid, frame_idx):
        st = self.data[tid]
        if st["locked"]:
            # ★ گام ۵ — بازبینیِ دوره‌ایِ افرادِ سبز.
            #   سناریویی که این را لازم می‌کند: کسی با صورتِ باز وارد
            #   می‌شود، سبز قفل می‌شود، بعد داخلِ مغازه ماسک می‌کشد.
            #   با قفلِ دائمی، سیستم دیگر هرگز نگاهش نمی‌کرد.
            return (frame_idx - st["locked_frame"]) >= self.GREEN_RECHECK_FRAMES
        if st["mode"] == "fast":
            return True

        # ★ C2 — آهنگِ بازبینیِ تطبیقی: بودجه به کسی برسد که مهم است.
        #
        #   در حال بررسی → هر فریم. باید سریع به حکم برسیم.
        #   قرمز         → هر فریم. مهم‌ترین فرد در صحنه است.
        #   نارنجی       → هر ۳ فریم. ماسکِ پزشکی رایج و پایدار است؛
        #                  بررسیِ مکررش اتلافِ بودجه است.
        #   سبزِ قفل‌شده → هر ۶۰ فریم (بالاتر).
        #
        #   نسخهٔ قبلی برای نارنجی و قرمز هر دو «هر ۲ فریم» بود — یعنی
        #   به فردِ خطرناک و فردِ بی‌خطر یک اندازه توجه می‌کرد.
        return frame_idx % self.CADENCE.get(st["color"], self.FOCUS_INTERVAL) == 0

    def register_vote(self, tid, category, conf, frame_idx=0):
        st = self.data[tid]
        st["just_finalized"] = None
        st["checks"] += 1                       # ★ گام ۶: شمارشِ بررسی‌ها

        # ★ گام ۵ — رأیِ بازبینی برای فردی که قبلاً سبز قفل شده بود.
        if st["locked"]:
            st["locked_frame"] = frame_idx      # ساعتِ بازبینی صفر شود
            if category == "green":
                return                          # هنوز صورتش باز است — کاری نکن
            # پوشش دیده شد → قفل می‌شکند و می‌رود به حالتِ فوکوس
            st["locked"] = False
            st["mode"] = "focus"
            st["votes"] = []
            st["focus_window"].clear()
            st["focus_window"].append(category)
            st["color"], st["label"] = category, LABELS[category]
            if category == "red":
                st["just_finalized"] = "red"
            return

        if st["mode"] == "fast":
            st["votes"].append((category, conf))
            if len(st["votes"]) >= self.FAST_VOTES_NEEDED:
                score = {"green": 0.0, "orange": 0.0, "red": 0.0}
                for c, cf in st["votes"]:
                    score[c] += cf
                best = max(score, key=score.get)
                # ★ A3 — سهمِ رأیِ برنده از کلِ امتیاز = ضریبِ اطمینان.
                #   قبلاً این عدد محاسبه می‌شد ولی دور ریخته می‌شد؛
                #   حالا هم روی تصویر نوشته و هم در JSON ثبت می‌شود.
                total = sum(score.values()) or 1.0
                st["conf"] = score[best] / total
                if best == "green":
                    self._lock(tid, "green", frame_idx)
                else:
                    st["mode"] = "focus"
                    st["color"], st["label"] = best, LABELS[best]
                    st["focus_window"].append(best)
                    if best == "red":
                        st["just_finalized"] = "red"
        else:
            st["focus_window"].append(category)
            window = list(st["focus_window"])
            green_count = window.count("green")
            if green_count >= self.FOCUS_GREEN_NEEDED:
                self._lock(tid, "green", frame_idx)
            else:
                sub = [c for c in window if c in ("orange", "red")]
                if sub:
                    # ★ C3 — تساویِ آرا دیگر تصادفی نیست.
                    #   `max(set(sub), key=sub.count)` وقتی دو رنگ تعدادِ
                    #   برابر دارند، هرکدام را که در پیمایشِ set زودتر
                    #   بیاید برمی‌گرداند — و ترتیبِ set تضمین‌شده نیست.
                    #   یعنی با ۲ نارنجی و ۲ قرمز، رنگ می‌توانست بی‌دلیل
                    #   بین دو حالت بپرد. حالا در تساوی رنگِ فعلی حفظ
                    #   می‌شود؛ فقط با اکثریتِ واقعی عوض می‌شود.
                    counts = {c: sub.count(c) for c in set(sub)}
                    top = max(counts.values())
                    winners = [c for c, n in counts.items() if n == top]
                    if len(winners) > 1 and st["color"] in winners:
                        best = st["color"]           # تساوی → همان‌که هست
                    else:
                        best = sorted(winners)[0]    # قطعی و تکرارپذیر
                    st["conf"] = counts[best] / len(window)   # ★ A3
                    if best == "red" and st["color"] != "red":
                        st["just_finalized"] = "red"
                    st["color"], st["label"] = best, LABELS[best]

    def _lock(self, tid, category, frame_idx=0):
        st = self.data[tid]
        st["locked"] = True
        st["locked_frame"] = frame_idx          # ★ گام ۵: شروعِ شمارشِ بازبینی
        st["color"] = category
        st["label"] = LABELS[category]

state_mgr = TrackStateManager()

---
## ۵ب) ★ C1 — حافظهٔ کوتاه‌مدتِ هویت

**مسئله:** فرد پشتِ قفسه می‌رود و برمی‌گردد → ByteTrack شناسهٔ تازه
می‌دهد → حکمِ قبلی از بین می‌رود. دزدِ قرمز بعد از دو ثانیه انسداد
دوباره «Analyzing» می‌شود.

**نگرانیِ درست: اگر دو نفر لباسِ شبیه داشته باشند چه؟** سه محافظ:

| محافظ | کار |
|---|---|
| حافظهٔ کوتاه | پیش‌فرض ۸ ثانیه. هرچه کوتاه‌تر، برخوردِ تصادفی کمتر |
| گیتِ مکانی | باید نزدیکِ محلِ ناپدیدشدن ظاهر شود و اندازهٔ جعبه هم‌خوان باشد |
| ★ آزمونِ حاشیه | اگر دو هویت **هر دو** شبیه باشند، هیچ‌کدام انتخاب نمی‌شود |

آزمونِ حاشیه مهم‌ترین است: در ابهام، سیستم ترجیح می‌دهد از صفر شروع
کند تا اینکه اشتباهی حکم را منتقل کند.

### و یک قانونِ ایمنی

> **هیچ‌وقت «قفل» به ارث نمی‌رسد.**

فردِ جدید رنگِ قبلی را نشان می‌دهد (پیوستگیِ بصری) ولی **دوباره
رأی‌گیری می‌شود**. اگر تطبیق اشتباه بوده باشد، ظرفِ چند فریم خودش را
اصلاح می‌کند.

اگر قفلِ سبز به ارث می‌رسید، یک تطبیقِ غلط می‌توانست یک دزد را برای
همیشه سبز کند — آن یک حفرهٔ امنیتی بود، نه یک اشکالِ کیفیت.

In [ ]:
# ---------------- 5b) ★ C1: حافظهٔ کوتاه‌مدتِ هویت --------------------
#
# مسئله: وقتی کسی پشتِ قفسه می‌رود و برمی‌گردد، ByteTrack شناسهٔ تازه
# می‌دهد. با شناسهٔ تازه، حکمِ قبلی از بین می‌رود و همه‌چیز از صفر
# شروع می‌شود — یعنی یک دزدِ قرمز بعد از یک انسدادِ دو ثانیه‌ای دوباره
# «Analyzing» می‌شود.
#
# سه محافظ در برابرِ اشتباه‌گرفتنِ افراد (مثلاً لباس‌های شبیه):
#
#   ۱) حافظهٔ کوتاه — پیش‌فرض ۸ ثانیه. هرچه بازه کوتاه‌تر، احتمالِ
#      برخوردِ تصادفیِ دو نفرِ شبیه کمتر.
#   ۲) گیتِ مکانی — فردِ جدید باید نزدیکِ جایی ظاهر شود که فردِ قبلی
#      ناپدید شده بود، و اندازهٔ جعبه‌اش هم هم‌خوان باشد.
#   ۳) ★ آزمونِ حاشیه — اگر دو هویتِ به‌یادمانده *هر دو* شبیه باشند
#      (مثلاً دو نفر با لباسِ هم‌رنگ)، هیچ‌کدام انتخاب نمی‌شود.
#      این مهم‌ترین محافظ است: در ابهام، سیستم ترجیح می‌دهد از صفر
#      شروع کند تا اینکه اشتباهی حکم را منتقل کند.
#
# و مهم‌تر از همه — قانونِ ایمنی:
#
#   ★ هیچ‌وقت «قفل» به ارث نمی‌رسد.
#     فردِ جدید رنگِ قبلی را نمایش می‌دهد (پیوستگیِ بصری) ولی
#     دوباره رأی‌گیری می‌شود. اگر تطبیق اشتباه بوده باشد، ظرفِ چند
#     فریم خودش را اصلاح می‌کند. اگر قفلِ سبز به ارث می‌رسید، یک
#     تطبیقِ غلط می‌توانست یک دزد را برای همیشه سبز کند.


class TrackMemory:
    def __init__(self, memory_seconds=8.0, fps=30.0,
                 match_th=0.80, margin=0.06, max_center_dist=0.35,
                 sig_every=5):
        self.ttl = int(memory_seconds * fps)
        self.match_th = match_th          # کمینهٔ شباهت برای پذیرش
        self.margin = margin              # فاصلهٔ لازم از دومین گزینه
        self.max_center_dist = max_center_dist   # نسبت به عرضِ فریم
        self.sig_every = sig_every
        self.slots = {}                   # tid -> {sig, box, frame, snap}
        self.stats = {"inherited": 0, "rejected_margin": 0, "rejected_far": 0}

    # ------------------------------------------------------------------
    @staticmethod
    def signature(crop):
        """امضای رنگیِ بدن — هیستوگرامِ Hue/Saturation. ارزان و ساده."""
        if crop is None or crop.size == 0:
            return None
        small = cv2.resize(crop, (48, 96), interpolation=cv2.INTER_AREA)
        hsv = cv2.cvtColor(small, cv2.COLOR_BGR2HSV)
        hist = cv2.calcHist([hsv], [0, 1], None, [24, 24], [0, 180, 0, 256])
        cv2.normalize(hist, hist, 0, 1, cv2.NORM_MINMAX)
        return hist.flatten()

    # ------------------------------------------------------------------
    def observe(self, tid, crop, box, frame_idx, st):
        """هر چند فریم یک بار، امضا و آخرین وضعیتِ فرد را به‌روز کن."""
        slot = self.slots.get(tid)
        need_sig = slot is None or (frame_idx - slot["frame"]) >= self.sig_every
        sig = self.signature(crop) if need_sig else slot["sig"]
        if sig is None:
            return
        self.slots[tid] = {
            "sig": sig, "box": tuple(float(v) for v in box), "frame": frame_idx,
            "snap": {"color": st["color"], "label": st["label"],
                     "mode": st["mode"], "conf": st.get("conf", 0.0)},
        }

    # ------------------------------------------------------------------
    def forget_old(self, frame_idx):
        dead = [t for t, s in self.slots.items()
                if (frame_idx - s["frame"]) > self.ttl]
        for t in dead:
            self.slots.pop(t, None)

    # ------------------------------------------------------------------
    def try_inherit(self, tid, st, crop, box, frame_idx, active_ids, frame_w):
        """
        اگر این شناسهٔ تازه به یکی از هویت‌های گم‌شده بخورد، رنگ و حالتش
        را به ارث می‌برد — ولی **قفل را نه**.
        """
        sig = self.signature(crop)
        if sig is None:
            return None

        cx = (box[0] + box[2]) / 2.0
        cy = (box[1] + box[3]) / 2.0
        bw = max(1.0, box[2] - box[0])
        limit = self.max_center_dist * frame_w

        scored = []
        for old_tid, slot in self.slots.items():
            if old_tid == tid or old_tid in active_ids:
                continue                                   # هنوز خودش فعال است
            if (frame_idx - slot["frame"]) > self.ttl:
                continue                                   # فراموش شده
            ob = slot["box"]
            ocx, ocy = (ob[0] + ob[2]) / 2.0, (ob[1] + ob[3]) / 2.0
            obw = max(1.0, ob[2] - ob[0])
            # ۲) گیتِ مکانی: هم فاصله، هم هم‌خوانیِ اندازه
            if ((cx - ocx) ** 2 + (cy - ocy) ** 2) ** 0.5 > limit:
                self.stats["rejected_far"] += 1
                continue
            if not (0.6 <= bw / obw <= 1.7):
                self.stats["rejected_far"] += 1
                continue
            score = float(cv2.compareHist(sig.reshape(24, 24),
                                          slot["sig"].reshape(24, 24),
                                          cv2.HISTCMP_CORREL))
            scored.append((score, old_tid))

        if not scored:
            return None
        scored.sort(reverse=True)
        best_score, best_tid = scored[0]
        if best_score < self.match_th:
            return None
        # ۳) آزمونِ حاشیه — اگر دومی هم تقریباً به همان خوبی است، رد کن
        if len(scored) > 1 and (best_score - scored[1][0]) < self.margin:
            self.stats["rejected_margin"] += 1
            return None

        snap = self.slots[best_tid]["snap"]
        # ★ قانونِ ایمنی: رنگ و حالت به ارث می‌رسد، قفل نه.
        st["color"] = snap["color"]
        st["label"] = snap["label"]
        st["conf"] = snap["conf"]
        st["locked"] = False
        st["mode"] = "focus" if snap["color"] in ("orange", "red") else "fast"
        st["votes"] = []
        self.slots.pop(best_tid, None)
        self.stats["inherited"] += 1
        return best_tid, best_score

---
## ۶) ابزارِ نمایش — گالری و اسکلت  ★ گام ۷

**★ گام ۷:** رنگِ پیش‌فرضِ `draw_upper_skeleton` دیگر زردِ ثابت نیست.
حالا فراخوان رنگِ وضعیتِ همان فرد را می‌فرستد، پس کادر و اسکلت و
برچسب هم‌رنگ‌اند.

In [ ]:
# ---------------- 6) Presentation Helpers (Gallery + Skeleton) --------
class PresentationGallery:
    """گالری تصاویر کوچک در گوشه تصویر - آخرین افراد شناسایی‌شده"""
    def __init__(self, max_items=4, thumb_size=140):
        self.items = deque(maxlen=max_items)   # هر آیتم: (img, label, color)
        self.thumb_size = thumb_size

    def add(self, crop_bgr, label, color):
        if crop_bgr is None or crop_bgr.size == 0:
            return
        thumb = cv2.resize(crop_bgr, (self.thumb_size, self.thumb_size))
        self.items.append((thumb, label, color))

    def draw(self, frame):
        h, w = frame.shape[:2]
        pad = 10
        for i, (thumb, label, color) in enumerate(self.items):
            x2 = w - pad
            x1 = x2 - self.thumb_size
            y1 = pad + i * (self.thumb_size + 35)
            y2 = y1 + self.thumb_size
            if y2 > h:
                break
            frame[y1:y2, x1:x2] = thumb
            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 3)
            cv2.putText(frame, label, (x1, y2 + 20),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 2)

gallery = PresentationGallery(max_items=4, thumb_size=140)

def draw_upper_skeleton(frame, kxy, kconf, color=(160, 160, 160), conf_th=0.4):
    """
    رسم اسکلتِ بالاتنه.

    ★ گام ۷ — رنگِ پیش‌فرض دیگر زردِ ثابت نیست. حالا فراخوان رنگِ
    وضعیتِ همان فرد را می‌فرستد، پس کادر و اسکلت و برچسب هم‌رنگ‌اند
    و کلِ فریم با یک نگاه خوانده می‌شود.
    """
    for a, b in UPPER_BODY_SKELETON:
        if kconf[a] < conf_th or kconf[b] < conf_th:
            continue
        pa = tuple(map(int, kxy[a]))
        pb = tuple(map(int, kxy[b]))
        cv2.line(frame, pa, pb, color, 2)
    for idx in [NOSE, LEYE, REYE, LEAR, REAR, LSHOULDER, RSHOULDER, LELBOW, RELBOW, LWRIST, RWRIST]:
        if kconf[idx] >= conf_th:
            p = tuple(map(int, kxy[idx]))
            cv2.circle(frame, p, 4, color, -1)

---
## ۷) خط لولهٔ اصلی

دو تابعِ تکراریِ نسخهٔ اصلی یکی شده‌اند. بررسی کردم — منطقِ تصمیم در
هر دو **یکسان** بود و نسخهٔ دمو فقط سه چیزِ نمایشی اضافه داشت:

| پرچم | پیش‌فرض | اثر |
|---|---|---|
| `show_skeleton` | `True` | رسمِ اسکلت (حالا هم‌رنگِ وضعیت) |
| `show_gallery` | `True` | گالریِ گوشهٔ تصویر |
| `slowmo_repeat` | `6` | تکرارِ فریم در لحظاتِ کلیدی (`1` = خاموش) |
| `min_eye_dist` | `8` | ★ گام ۶ |

**★ گام ۷ — ترتیبِ رسم عوض شد.** در نسخهٔ اصلی اسکلت در حلقهٔ *اول*
کشیده می‌شد، جایی که هنوز رنگِ وضعیت معلوم نبود. حالا کی‌پوینت‌ها در
`kpts_by_tid` نگه داشته می‌شوند و اسکلت در حلقهٔ *رسم* — بعد از
مشخص‌شدنِ رنگ — کشیده می‌شود.

**★ گام ۶ — برچسبِ پیشرفت.** در حالتِ خاکستری به‌جای
`Analyzing...` خالی، حالا `Analyzing... (2/3)` نوشته می‌شود؛ و اگر
هنوز هیچ بررسی‌ای ممکن نبوده `Analyzing... (too far)`. این‌طور معلوم
است سیستم فرد را دیده و دارد رویش کار می‌کند.

In [ ]:
# ---------------- 7) Main Pipeline ------------------------------------
def process_video(input_path, output_path, conf_thres=0.4, yolo_imgsz=640,
                  face_conf_th=0.5, slowmo_repeat=6,
                  show_skeleton=True, show_gallery=True, min_eye_dist=8,
                  track_memory=None):
    cap = cv2.VideoCapture(input_path)
    if not cap.isOpened():
        print("❌ Error: Cannot open video")
        return
    fps = cap.get(cv2.CAP_PROP_FPS) or 25
    W = int(cap.get(cv2.CAP_PROP_FRAME_WIDTH))
    H = int(cap.get(cv2.CAP_PROP_FRAME_HEIGHT))
    total = int(cap.get(cv2.CAP_PROP_FRAME_COUNT))
    cap.release()

    out = cv2.VideoWriter(output_path, cv2.VideoWriter_fourcc(*"mp4v"), fps, (W, H))
    if not out.isOpened():                                        # [ایمنی]
        raise RuntimeError(f"❌ فایل خروجی باز نشد: {output_path}")
    t0 = time.time()
    last_pct = -1
    frame_idx = 0
    events = []                      # ★ A3: لاگِ رویدادها
    seen_ids = set()   # برای تشخیص "فرد کاملا جدید" جهت افکت اسلوموشن

    results_gen = pose_model.track(
        source=input_path, classes=[0], conf=conf_thres, imgsz=yolo_imgsz,
        tracker="bytetrack.yaml", stream=True, verbose=False, persist=True,
        half=USE_HALF
    )

    for r in results_gen:
        frame_idx += 1
        frame = r.orig_img

        if r.boxes.id is None or r.keypoints is None:
            out.write(frame)
            pct = int(frame_idx / total * 100) if total > 0 else 0
            if pct != last_pct and pct % 5 == 0:
                print(f"⏳ Progress: {pct}%"); last_pct = pct
            continue

        ids = r.boxes.id.int().cpu().tolist()
        boxes = r.boxes.xyxy.cpu().numpy()
        kpts_xy_all = r.keypoints.xy.cpu().numpy()
        kpts_conf_all = r.keypoints.conf.cpu().numpy() if r.keypoints.conf is not None else np.ones(kpts_xy_all.shape[:2])

        batch_crops, batch_meta = [], []
        trigger_slowmo = False   # آیا این فریم باید کند نمایش داده بشه؟
        kpts_by_tid = {}         # ★ گام ۷: برای رسمِ اسکلت با رنگِ وضعیت
        if track_memory is not None and frame_idx % 30 == 0:
            track_memory.forget_old(frame_idx)   # ★ C1: فراموشیِ دوره‌ای

        for tid, box, kxy, kconf in zip(ids, boxes, kpts_xy_all, kpts_conf_all):
            x1, y1, x2, y2 = [int(v) for v in box]
            x1, y1 = max(0, x1), max(0, y1)
            x2, y2 = min(W, x2), min(H, y2)
            person_crop = frame[y1:y2, x1:x2]
            if person_crop.size == 0:
                continue

            # ★ C1 — اگر شناسه تازه است، شاید همان کسی باشد که چند
            #   ثانیه پیش پشتِ قفسه گمش کردیم.
            is_new_track = tid not in state_mgr.data
            st = state_mgr.ensure(tid)
            if track_memory is not None:
                if is_new_track:
                    track_memory.try_inherit(tid, st, person_crop, box,
                                             frame_idx, ids, W)
                track_memory.observe(tid, person_crop, box, frame_idx, st)

            kpts_by_tid[tid] = (kxy, kconf)          # ★ گام ۷

            # --- افکت نمایشی: فرد کاملا جدید -> اسلوموشن + اضافه به گالری ---
            if tid not in seen_ids:
                seen_ids.add(tid)
                trigger_slowmo = True
                square_crop = person_crop.copy()
                gallery.add(square_crop, f"New Person ID {tid}", (255, 255, 0))

            if state_mgr.should_analyze(tid, frame_idx):
                local_kxy = kxy.copy()
                local_kxy[:, 0] -= x1
                local_kxy[:, 1] -= y1

                face_crop, face_score = align_and_crop_face(
                    person_crop, local_kxy, kconf, face_conf_th, min_eye_dist)

                if face_crop is not None:
                    batch_crops.append(face_crop)
                    batch_meta.append(tid)

        if batch_crops:
            results = classify_mask_batch(batch_crops)
            for (tid, (mask_label, conf), face_bgr) in zip(batch_meta, results, batch_crops):
                if mask_label == "no_mask":
                    state_mgr.register_vote(tid, "green", conf, frame_idx)
                else:
                    cat = "red" if is_suspicious(face_bgr) else "orange"
                    state_mgr.register_vote(tid, cat, conf, frame_idx)

        # ---- ★ A2: برشِ سوژه، از فریمی که هنوز رویش رسم نشده ----
        for tid, box in zip(ids, boxes):
            st = state_mgr.ensure(tid)
            if st.get("just_finalized") == "red":
                trigger_slowmo = True
                x1c, y1c = max(0, int(box[0])), max(0, int(box[1]))
                x2c, y2c = min(W, int(box[2])), min(H, int(box[3]))
                if x2c > x1c and y2c > y1c:
                    gallery.add(frame[y1c:y2c, x1c:x2c].copy(),
                                f"SUSPECT ID {tid}", (0, 0, 255))
                events.append({                       # ★ A3
                    "frame": frame_idx,
                    "time_s": round(frame_idx / fps, 2),
                    "track_id": int(tid),
                    "state": "red",
                    "label": LABELS["red"],
                    "confidence": round(float(st.get("conf", 0.0)), 3),
                    "checks": int(st.get("checks", 0)),
                })
                st["just_finalized"] = None

        # ---- رسم باکس‌ها ----
        for tid, box in zip(ids, boxes):
            x1, y1, x2, y2 = [int(v) for v in box]
            st = state_mgr.ensure(tid)
            color = COLORS[st["color"]]

            # ★ گام ۷ — اسکلت با رنگِ وضعیتِ همین فرد
            if show_skeleton and tid in kpts_by_tid:
                kxy_d, kconf_d = kpts_by_tid[tid]
                draw_upper_skeleton(frame, kxy_d, kconf_d, color)

            # ★ گام ۶ — در حالتِ «در حال بررسی» پیشرفت را نشان بده تا
            #   معلوم باشد سیستم فرد را دیده و دارد رویش کار می‌کند،
            #   نه اینکه او را نادیده گرفته باشد.
            # ★ A3 — ضریبِ اطمینان روی تصویر. تا حالا محاسبه می‌شد
            #   ولی هیچ‌جا دیده نمی‌شد؛ حالا هم روی کادر است و هم در JSON.
            label = st["label"]
            if st["color"] != "gray" and st.get("conf", 0) > 0:
                label = f"{label} {st['conf']:.0%}"
            if st["color"] == "gray":
                got = len(st["votes"])
                if st["checks"] == 0:
                    label = "Analyzing... (too far)"
                else:
                    label = f"Analyzing... ({got}/{state_mgr.FAST_VOTES_NEEDED})"

            cv2.rectangle(frame, (x1, y1), (x2, y2), color, 2)
            cv2.putText(frame, f"ID {tid}: {label}", (x1, max(20, y1 - 8)),
                        cv2.FONT_HERSHEY_SIMPLEX, 0.55, color, 2)

            if st["color"] == "red":
                cv2.putText(frame, "ALERT!", (x1, y2 + 22),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 255), 2)


        # ---- گالری گوشه تصویر ----
        if show_gallery:
            gallery.draw(frame)

        # ---- نوشتن خروجی (با افکت اسلوموشن در لحظات کلیدی) ----
        repeat = slowmo_repeat if trigger_slowmo else 1
        for _ in range(repeat):
            out.write(frame)

        pct = int(frame_idx / total * 100) if total > 0 else 0
        if pct != last_pct and pct % 5 == 0:
            print(f"⏳ Progress: {pct}%")
            last_pct = pct

    out.release()

    # ---- ★ A3: وضعیتِ نهاییِ هر فرد هم ثبت شود ----
    for tid, st in state_mgr.data.items():
        events.append({
            "frame": frame_idx,
            "time_s": round(frame_idx / fps, 2),
            "track_id": int(tid),
            "state": st["color"],
            "label": st["label"],
            "confidence": round(float(st.get("conf", 0.0)), 3),
            "checks": int(st.get("checks", 0)),
            "final": True,
        })

    json_path = os.path.splitext(output_path)[0] + "_events.json"
    with open(json_path, "w", encoding="utf-8") as f:
        json.dump({"video": input_path,
                   "frames": frame_idx,
                   "pose_weights": POSE_WEIGHTS,
                   "params": {"conf_thres": conf_thres,
                              "yolo_imgsz": yolo_imgsz,
                              "face_conf_th": face_conf_th,
                              "min_eye_dist": min_eye_dist},
                   "events": events}, f, ensure_ascii=False, indent=2)

    print(f"✅ Done in {time.time() - t0:.2f}s -> {output_path}")
    print(f"📄 {len(events)} رویداد -> {json_path}")
    return events

---
## ۸) اجرا

### اتصال Google Drive

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### تنظیم مسیرها و اجرا

In [ ]:
# ---------------- مسیرها ----------------
INPUT_VIDEO  = "/content/drive/MyDrive/9.mp4"     # ← ویدیوی خودت
OUTPUT_VIDEO = "/content/output_v4.mp4"

# ---------------- پارامترها (همان مقادیرِ نسخهٔ اصلی) ----------------
CONF_THRES    = 0.4
YOLO_IMGSZ    = 640
FACE_CONF_TH  = 0.5

# ---------------- ★ گام ۶ ----------------
MIN_EYE_DIST  = 8       # حداقلِ فاصلهٔ دو چشم برای صدورِ حکم

# ---------------- ★ C1: حافظهٔ هویت ----------------
USE_TRACK_MEMORY = True
MEMORY_SECONDS   = 8.0   # ۵ تا ۱۰ منطقی است. کوتاه‌تر = محتاط‌تر
MEMORY_MATCH_TH  = 0.80  # کمینهٔ شباهتِ رنگِ لباس (۰..۱)
MEMORY_MARGIN    = 0.06  # ★ اگر دومین گزینه از این نزدیک‌تر بود → رد
MEMORY_MAX_DIST  = 0.35  # حداکثر جابه‌جایی، نسبت به عرضِ فریم

# ---------------- جلوه‌های نمایشی ----------------
SHOW_SKELETON = True
SHOW_GALLERY  = True
SLOWMO_REPEAT = 6       # ۱ = خاموش

import os
if not os.path.exists(INPUT_VIDEO):                               # [ایمنی]
    raise FileNotFoundError(f"❌ ویدیو پیدا نشد: {INPUT_VIDEO}")

# حالتِ داخلی را قبل از هر اجرا صفر می‌کنیم — وگرنه اگر سلول را دو بار
# اجرا کنی، شناسه‌ها و رأی‌های اجرای قبلی باقی می‌مانند.
state_mgr = TrackStateManager()
gallery   = PresentationGallery(max_items=4, thumb_size=140)

# ★ C1 — حافظه هم باید هر اجرا صفر شود
_fps_guess = cv2.VideoCapture(INPUT_VIDEO).get(cv2.CAP_PROP_FPS) or 30.0
track_memory = TrackMemory(memory_seconds=MEMORY_SECONDS,
                           fps=_fps_guess,
                           match_th=MEMORY_MATCH_TH,
                           margin=MEMORY_MARGIN,
                           max_center_dist=MEMORY_MAX_DIST) if USE_TRACK_MEMORY else None

# ★ گام ۵ — نرخِ بازبینیِ افرادِ سبز (۶۰ فریم ≈ ۲ ثانیه در ۳۰fps)
state_mgr.GREEN_RECHECK_FRAMES = 60

events = process_video(INPUT_VIDEO, OUTPUT_VIDEO,
                       conf_thres=CONF_THRES,
                       yolo_imgsz=YOLO_IMGSZ,
                       face_conf_th=FACE_CONF_TH,
                       slowmo_repeat=SLOWMO_REPEAT,
                       show_skeleton=SHOW_SKELETON,
                       show_gallery=SHOW_GALLERY,
                       min_eye_dist=MIN_EYE_DIST,
                       track_memory=track_memory)

### نمایش ویدیوی خروجی

In [ ]:
import subprocess, os
from base64 import b64encode
from IPython.display import HTML

web = "/content/preview.mp4"
subprocess.run(["ffmpeg", "-y", "-loglevel", "error", "-i", OUTPUT_VIDEO,
                "-vcodec", "libx264", "-crf", "26", web], check=False)

path = web if os.path.exists(web) else OUTPUT_VIDEO
data = b64encode(open(path, "rb").read()).decode()
HTML(f'<video width=860 controls>'
     f'<source src="data:video/mp4;base64,{data}" type="video/mp4"></video>')

### گزارشِ وضعیتِ نهاییِ هر فرد

بعد از اجرا، این سلول می‌گوید سیستم دربارهٔ هر نفر به چه نتیجه‌ای
رسید و چند بار بررسی‌اش کرد — برای راستی‌آزمایی مفید است.

In [ ]:
from collections import Counter

rows = []
for tid, st in sorted(state_mgr.data.items()):
    rows.append((tid, st["color"], st["checks"], st["locked"], st["mode"]))

print(f"{'ID':>4}  {'وضعیت':<8} {'بررسی':>6} {'قفل':>5}  حالت")
print("-" * 44)
for tid, color, checks, locked, mode in rows:
    print(f"{tid:>4}  {color:<8} {checks:>6} {str(locked):>5}  {mode}")

print("\nجمع‌بندی:", dict(Counter(r[1] for r in rows)))
print("مجموع بررسی‌ها:", sum(r[2] for r in rows))

---
## ★ مقایسهٔ YOLO11s با YOLO26

مدلِ ژست فقط یک متغیر است، پس مقایسه ساده است: این سلول همان ویدیو را
با هر دو مدل روی چند صد فریمِ اول اجرا می‌کند و نتیجه را کنار هم
می‌گذارد.

**چه چیزی را مقایسه کن:**

| معیار | یعنی چه |
|---|---|
| تعدادِ بررسی | کی‌پوینتِ بهتر → گیت بیشتر باز می‌شود → بررسیِ بیشتر |
| توزیعِ رنگ‌ها | چند نفر به حکمِ قطعی رسیدند و چند نفر «Analyzing» ماندند |
| میانگینِ اطمینان | حکم‌ها قاطع‌ترند یا مرزی |
| زمان | هزینهٔ سرعت |

**⚠️ نکتهٔ مهم:** YOLO26 از RLE برای مکان‌یابیِ کی‌پوینت استفاده می‌کند،
که یعنی *کالیبراسیونِ اطمینان* ممکن است فرق کند. گیتِ ما
(`FACE_CONF_TH = 0.5`) دقیقاً روی همان اطمینان‌ها کار می‌کند.

اگر با YOLO26 دیدی خیلی‌ها «Analyzing» ماندند، مدل بد نیست — آستانه
باید جابه‌جا شود. `FACE_CONF_TH` را روی ۰.۴ و ۰.۶ هم امتحان کن.

In [ ]:
# مقایسهٔ دو مدل روی همان ویدیو — چند صد فریمِ اول کافی است
import time
from collections import Counter

COMPARE_MODELS = ["yolo11s-pose.pt", "yolo26s-pose.pt"]

results = {}
for w in COMPARE_MODELS:
    print()
    print("=" * 60)
    print("  " + w)
    print("=" * 60)
    try:
        pose_model = YOLO(w)
    except Exception as e:
        print("  bargozari nashod:", type(e).__name__, e)
        print("  (shayad ultralytics ghadimi ast:  !pip install -U ultralytics)")
        continue

    POSE_WEIGHTS = w
    state_mgr = TrackStateManager()
    gallery   = PresentationGallery(max_items=4, thumb_size=140)
    tmem = None
    if USE_TRACK_MEMORY:
        tmem = TrackMemory(memory_seconds=MEMORY_SECONDS, fps=_fps_guess,
                           match_th=MEMORY_MATCH_TH, margin=MEMORY_MARGIN,
                           max_center_dist=MEMORY_MAX_DIST)

    t0 = time.time()
    ev = process_video(INPUT_VIDEO, "/content/cmp_" + w.replace(".pt", "") + ".mp4",
                       conf_thres=CONF_THRES, yolo_imgsz=YOLO_IMGSZ,
                       face_conf_th=FACE_CONF_TH, slowmo_repeat=1,
                       show_skeleton=False, show_gallery=False,
                       min_eye_dist=MIN_EYE_DIST, track_memory=tmem)
    dt = time.time() - t0

    finals = [e for e in ev if e.get("final")]
    confs  = [e["confidence"] for e in finals if e["confidence"] > 0]
    results[w] = {
        "seconds": dt,
        "tracks": len(finals),
        "checks": sum(e["checks"] for e in finals),
        "colors": dict(Counter(e["state"] for e in finals)),
        "mean_conf": (sum(confs) / len(confs)) if confs else 0.0,
        "alerts": sum(1 for e in ev if (not e.get("final")) and e["state"] == "red"),
    }

print()
print()
print("=" * 76)
print("  model                seconds   tracks   checks   alerts   mean-conf")
print("=" * 76)
for w, r in results.items():
    print("  {:<20}{:>8.1f}{:>9}{:>9}{:>9}{:>11.0%}".format(
        w, r["seconds"], r["tracks"], r["checks"], r["alerts"], r["mean_conf"]))
    print("    colors:", r["colors"])
print("=" * 76)
print()
print("agar YOLO26 hoshdar-haye dorost-tar va etminan-e balatar dad -> negahash dar:")
print('   POSE_WEIGHTS = "yolo26s-pose.pt"   va sellul-e model ra dobare Run kon')


### ذخیره در Google Drive

In [ ]:
import shutil, os

DRIVE_DEST = '/content/drive/MyDrive/output_v4_saved.mp4'
if os.path.exists(OUTPUT_VIDEO):
    shutil.copy(OUTPUT_VIDEO, DRIVE_DEST)
    print(f"✅ ذخیره شد: {DRIVE_DEST}")
else:
    print("❌ فایل خروجی پیدا نشد — اول سلولِ اجرا را ران کن.")